# PM2.5 Data Validation (Excel Cleaned Version)
Checking structure, missing values, duplicates, and ranges.

This notebook validates Excel-cleaned datasets. Final pipeline uses raw hourly NAPS data aggregated to daily city-level in Python for reproducibility.


In [8]:
import pandas as pd
import numpy as np
import os
pm25 = pd.read_csv("../../data/master/PM25_2020 cleaned Master_20_23.csv")
o3   = pd.read_csv("../../data/master/O3_2020 cleaned Master_20_23.csv")
print("PM25 shape:", pm25.shape)
print("O3 shape:", o3.shape)

PM25 shape: (386424, 6)
O3 shape: (338222, 6)


In [10]:
print(pm25.columns)
print(o3.columns)

Index(['City', 'Date', 'PM25', 'Year', 'Month', 'Season'], dtype='object')
Index(['City', 'Date', 'O3', 'Year', 'Month', 'Season'], dtype='object')


In [14]:
pm25["Date"] = pd.to_datetime(pm25["Date"], errors="coerce")
o3["Date"] = pd.to_datetime(o3["Date"], errors="coerce")
print("Invalid PM25 dates:", pm25["Date"].isna().sum())
print("Invalid O3 dates:", o3["Date"].isna().sum())     

Invalid PM25 dates: 0
Invalid O3 dates: 0


In [16]:
print("PM25 missing:\n", pm25.isna().sum())
print("\nO3 missing:\n", o3.isna().sum())

PM25 missing:
 City      0
Date      0
PM25      0
Year      0
Month     0
Season    0
dtype: int64

O3 missing:
 City      0
Date      0
O3        0
Year      0
Month     0
Season    0
dtype: int64


In [18]:
print("PM25 duplicates:", pm25.duplicated(subset=["Date","City"]).sum())
print("O3 duplicates:", o3.duplicated(subset=["Date","City"]).sum())

PM25 duplicates: 121983
O3 duplicates: 63188


In [20]:
duplicate_rows = pm25[pm25.duplicated(subset=["Date","City"], keep=False)]

duplicate_rows.sort_values(["City","Date"]).head(20)

,City,Date,PM25,Year,Month,Season
33672,Anzac,2020-01-01,5.916666667,2020,1,Winter
92598,Anzac,2020-01-01,,2020,1,Winter
33673,Anzac,2020-01-02,6.75,2020,1,Winter
92599,Anzac,2020-01-02,,2020,1,Winter
33674,Anzac,2020-01-03,1.75,2020,1,Winter
92600,Anzac,2020-01-03,,2020,1,Winter
33675,Anzac,2020-01-04,3.416666667,2020,1,Winter
92601,Anzac,2020-01-04,,2020,1,Winter
33676,Anzac,2020-01-05,4.791666667,2020,1,Winter
92602,Anzac,2020-01-05,,2020,1,Winter


In [24]:
pm25 = pm25.dropna(subset=["PM25"])
o3 = o3.dropna(subset=["O3"])

In [26]:
pm25 = pm25.drop_duplicates(subset=["City","Date"])
o3   = o3.drop_duplicates(subset=["City","Date"])

In [28]:
print("PM25 duplicates:", pm25.duplicated(subset=["City","Date"]).sum())
print("O3 duplicates:", o3.duplicated(subset=["City","Date"]).sum())

print("New PM25 shape:", pm25.shape)
print("New O3 shape:", o3.shape)

PM25 duplicates: 0
O3 duplicates: 0
New PM25 shape: (264441, 6)
New O3 shape: (275034, 6)


In [32]:
print(pm25["PM25"].dtype)
print(o3["O3"].dtype)


object
object


In [34]:
pm25["PM25"] = pd.to_numeric(pm25["PM25"], errors="coerce")
o3["O3"]     = pd.to_numeric(o3["O3"], errors="coerce")

print("PM25 NaNs after conversion:", pm25["PM25"].isna().sum())
print("O3 NaNs after conversion:", o3["O3"].isna().sum())


PM25 NaNs after conversion: 17467
O3 NaNs after conversion: 8748


In [36]:
pm25 = pm25.dropna(subset=["PM25"])
o3   = o3.dropna(subset=["O3"])


In [38]:
pm25_cityday = pm25.groupby(
    ["City", "Date", "Year", "Month", "Season"],
    as_index=False
)["PM25"].mean()

o3_cityday = o3.groupby(
    ["City", "Date", "Year", "Month", "Season"],
    as_index=False
)["O3"].mean()


In [40]:
print("PM25 city-day shape:", pm25_cityday.shape)
print("O3 city-day shape:", o3_cityday.shape)

print("PM25 duplicates:", pm25_cityday.duplicated(subset=["City","Date"]).sum())
print("O3 duplicates:", o3_cityday.duplicated(subset=["City","Date"]).sum())


PM25 city-day shape: (246974, 6)
O3 city-day shape: (266286, 6)
PM25 duplicates: 0
O3 duplicates: 0
